In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv("../data/raw/House_Prices.csv")
df = df.iloc[:1460].copy()

In [3]:
df["TotalArea"] = df[["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]].sum(axis=1, skipna=True)

df["TotalBathrooms"] = (
    df["FullBath"].fillna(0)
    + 0.5 * df["HalfBath"].fillna(0)
    + df["BsmtFullBath"].fillna(0)
    + 0.5 * df["BsmtHalfBath"].fillna(0)
)

df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

df["TotalPorchArea"] = (
    df["WoodDeckSF"]
    + df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

df.loc[df["HouseAge"] < 0, "HouseAge"] = np.nan
df.loc[df["YearsSinceRemodel"] < 0, "YearsSinceRemodel"] = np.nan

In [4]:
# remove outliers that has higher GrLivArea with less saleprice
df = df.drop(index=[1298, 523])

df = df.drop(columns="Id")
df["MSSubClass"] = df["MSSubClass"].astype("str")

# fill missing values in categorical columns with none
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageQual", "GarageCond", "GarageFinish", "GarageType",
    "BsmtCond", "BsmtQual", "BsmtFinType1"
]
df[none_cols] = df[none_cols].fillna("None")

# fill missing values in num columns with 0
zero_cols = [
    "BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1",
    "GarageArea", "GarageCars"
]
df[zero_cols] = df[zero_cols].fillna(0)

# if the garage built year greater than the sold year mark the values as missing
df.loc[df["GarageYrBlt"] > df["YrSold"], "GarageYrBlt"] = np.nan


In [5]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [6]:
numeric_cols = X.select_dtypes(include="number").columns
categorical_cols = X.select_dtypes(exclude="number").columns

In [7]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [8]:
gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

In [9]:
mlflow.set_tracking_uri(
    "sqlite:////home/elgmouri/projects/aqari/mlflow.db"
)
mlflow.set_experiment("Aqari Model Comparison")
with mlflow.start_run(run_name="Gradient Boosting Cross Validation"):
    r2_scores = cross_val_score(
        gradient_boosting_pipeline,
        X,
        y,
        cv=5,
        scoring="r2"
    )
    mae_scores = -cross_val_score(
        gradient_boosting_pipeline,
        X,
        y,
        cv=5,
        scoring="neg_mean_absolute_error"
    )
    rmse_scores = -cross_val_score(
        gradient_boosting_pipeline,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )


    mlflow.log_params(
        gradient_boosting_pipeline.named_steps["model"].get_params()
    )

    mlflow.log_metrics({
        "mean_r2": r2_scores.mean(),
        "std_r2": r2_scores.std(),
        "mean_mae": mae_scores.mean(),
        "std_mae": mae_scores.std(),
        "mean_rmse": rmse_scores.mean(),
        "std_rmse": rmse_scores.std()
    })


In [10]:
print("R² scores:", r2_scores)
print("Mean R²:", r2_scores.mean())
print("Std R²:", r2_scores.std())

print("MAE scores:", mae_scores)
print("Mean MAE:", mae_scores.mean())
print("Std MAE:", mae_scores.std())

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

R² scores: [0.90973175 0.93172618 0.91324587 0.91965046 0.92057128]
Mean R²: 0.9189851090836332
Std R²: 0.0075346783105231215
MAE scores: [15156.39127226 14705.62959724 15334.61870167 13669.06293418
 15485.38493352]
Mean MAE: 14870.217487773585
Std MAE: 655.1116460598174
RMSE scores: [22347.72442093 21194.00454837 25640.36694798 20198.41988993
 23114.79231021]
Mean RMSE: 22499.06162348309
Std RMSE: 1858.276229446666


The 5 fold cross validation confirms that Gradient Boosting performs consistently across different data splits. The mean R² is approximately 0.919 with a very low standard deviation, indicating stable and reliable performance. The cross validation results are close to the original holdout evaluation, so the model’s performance does not appear to be due to a lucky train/test split.